In [1]:
import pandas as pd
import numpy as np
from nltk import word_tokenize
import os
from ast import literal_eval
from nltk.util import ngrams
from nltk.corpus import stopwords
stop=stopwords.words('english')
import spacy
nlp=spacy.load('en_core_web_sm')
from tqdm import tqdm

/Users/mstudio/miniconda3/envs/py3.10/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path='/Users/mstudio/Library/CloudStorage/Box-Box/coolie/sent/'
df=pd.read_csv(path+'/coolie.csv', converters={'state':literal_eval, 'context':literal_eval, 'city':literal_eval})

In [14]:
def get_ngrams(text, n):
    entire=[]
    n_grams=ngrams(word_tokenize(text), n)
    for i in n_grams:
        entire.append(i)
    return entire

In [15]:
df['context_str']=df['context'].apply(' '.join)

"san i'iuxcisco, aug. !'!!. atlt ices from\nslngnporo state that tho steamer namchow\nsailed from that port with 800 chinese coolies\nfor tho i'euang market. whllo at sea cholera broke out among tho\nchinese. the sick crowded lu the cabins,\nwhere n presbyterian minister and ono lady\npassenger had tnkun reluge."

In [16]:
def lemmatization(dataframe:pd.DataFrame()):
    dataframe['stopword']=dataframe['context_str'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))
    dataframe['punct']=dataframe['stopword'].str.replace('[^\w\s]','')
    dataframe['punct']=dataframe['punct'].str.replace('\n', '')
    dataframe['lemma']=dataframe['punct'].apply(lambda row: ' '.join([w.lemma_ for w in nlp(row)]))
    return dataframe

In [20]:
def reuse_df(data:pd.DataFrame()):
    no_reuse=[]
    pair_index1=[]
    pair_index2=[]
    pair_date1=[]
    pair_date2=[]
    pair_text1=[]
    pair_text2=[]
    pair_state1=[]
    pair_state2=[]
    for k in range(0,data.shape[0]):
        t=0
        for p in range(t,data.shape[0]):
            t+=1
            if t>k:
                if (k==p) & (len(list(set(data['5-gram'][k]).intersection(data['5-gram'][p]))) == len(data['5-gram'][k])):
                    pass
                elif len(list(set(data['5-gram'][k]).intersection(data['5-gram'][p]))) == 0:
                    pass
                elif len(list(set(data['5-gram'][k]).intersection(data['5-gram'][p]))) >= 3:
                    no_reuse.append(len(list(set(data['5-gram'][k]).intersection(data['5-gram'][p]))))
                    pair_index1.append(k) 
                    pair_index2.append(p)
                    pair_date1.append(data.iloc[k]['date'])
                    pair_date2.append(data.iloc[p]['date'])
                    pair_text1.append(data.iloc[k]['lemma'])
                    pair_text2.append(data.iloc[p]['lemma'])
                    pair_state1.append(data.iloc[k]['state'])
                    pair_state2.append(data.iloc[p]['state'])
    return pd.DataFrame({'no_reuse':no_reuse, 'pair_index1':pair_index1, 'pair_index2':pair_index2, 
             'pair_date1':pair_date1, 'pair_date2':pair_date2, 
              'pair_text1':pair_text1, 'pair_text2':pair_text2,
                'pair_state1':pair_state1, 'pair_state2':pair_state2})

In [18]:
df=lemmatization(df)

In [12]:
path='/Users/mstudio/Library/CloudStorage/Box-Box/coolie/sent/'
df=pd.read_csv(path+'/coolie-lemma.csv', converters={'state':literal_eval, 'context':literal_eval, 'city':literal_eval})

In [15]:
df['5-gram']=df['context_str'].apply(lambda x: get_ngrams(x, 5))

In [21]:
reprintdf=reuse_df(df)

In [ ]:
reprintdf.to_csv(path+'reprint.csv', index=False)